<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/IA_desaparece_persona_en_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --no-deps ultralytics simple-lama-inpainting
!pip install ultralytics-thop ultralytics-platform fire

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 11.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [simple-lama-inpainting]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics-thop]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
simple-lama-inpainting 0.1.2 requires fire<0.6.0,>=0.5.0, but you have fire 0.7.1 which is incompatible.
simple-lama-inpainting 0.1.2 requires numpy<2.0.0,>=1.24.3, but you have numpy 2.4.6 which is incompatible.
simple-lama-inpainting 0.1.2 requires opencv-python<5.0.0.0,>=4.8.0.74, but you have opencv-python 5.0.0.93 which is incompatible.
simple-lama-inpainting 0.1.2 requires pillow<10.0.0,>=9.5.0, but you have pillow 11.3.0 which is incompatible.


In [ ]:
!wget -q -O video_entrada.mp4 "https://github.com"
print("¡Entorno y video listos!")

¡Entorno y video listos!


In [ ]:
# Descarga un video real de personas caminando desde una fuente directa y estable en GitHub
!wget -q -O video_entrada.mp4 "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/people-detection.mp4"

import os
if os.path.exists("video_entrada.mp4") and os.path.getsize("video_entrada.mp4") > 1000:
    print("¡Video descargado y listo! Tamaño:", os.path.getsize("video_entrada.mp4") / (1024*1024), "MB")
else:
    print("Error al descargar el video. Por favor, intenta ejecutar la celda de nuevo.")

¡Video descargado y listo! Tamaño: 5.228594779968262 MB


In [ ]:
import cv2
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO
from simple_lama_inpainting import SimpleLama

# Cargar modelo de segmentación de YOLOv8 y modelo LaMa
modelo_yolo = YOLO("yolov8n-seg.pt")
simple_lama = SimpleLama()

print("Modelos cargados correctamente en:", "GPU" if torch.cuda.is_available() else "CPU")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


Downloading: "https://github.com/enesmsahin/simple-lama-inpainting/releases/download/v0.1.0/big-lama.pt" to /root/.cache/torch/hub/checkpoints/big-lama.pt
100%|██████████| 196M/196M [00:01<00:00, 139MB/s]
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Modelos cargados correctamente en: GPU


In [ ]:
cap = cv2.VideoCapture("video_entrada.mp4")

ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Guardaremos dos videos: el limpio y uno dividido (Antes vs Después) para tu TikTok
out_limpio = cv2.VideoWriter("video_limpio.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))
out_comparativo = cv2.VideoWriter("comparativo_tiktok.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto * 2))

kernel = np.ones((15, 15), np.uint8)  # Para dilatar la máscara y borrar sombras/bordes
max_frames = 150  # Procesa los primeros 5 segundos (cambia a None para el video completo)
frame_actual = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or (max_frames and frame_actual >= max_frames):
        break

    # 1. Detectar personas (clase 0 en COCO)
    resultados = modelo_yolo(frame, classes=[0], verbose=False)[0]
    mascara_total = np.zeros((alto, ancho), dtype=np.uint8)

    if resultados.masks is not None:
        for mask in resultados.masks.data:
            # Redimensionar máscara al tamaño original del frame
            m = mask.cpu().numpy()
            m = cv2.resize(m, (ancho, alto))
            mascara_total = np.maximum(mascara_total, (m > 0.5).astype(np.uint8) * 255)

        # Dilatar máscara para cubrir bordes del cuerpo
        mascara_total = cv2.dilate(mascara_total, kernel, iterations=2)

        # 2. Reconstruir el fondo con LaMa Inpainting
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resultado_pil = simple_lama(Image.fromarray(frame_rgb), Image.fromarray(mascara_total))
        frame_limpio = cv2.cvtColor(np.array(resultado_pil), cv2.COLOR_RGB2BGR)
        # Asegurar dimensiones exactas
        frame_limpio = cv2.resize(frame_limpio, (ancho, alto))
    else:
        frame_limpio = frame.copy()

    # Guardar frame limpio y frame dividido verticalmente (arriba original, abajo limpio)
    out_limpio.write(frame_limpio)
    combinado = np.vstack((frame, frame_limpio))
    out_comparativo.write(combinado)

    frame_actual += 1
    if frame_actual % 30 == 0:
        print(f"Procesados {frame_actual} fotogramas...")

cap.release()
out_limpio.release()
out_comparativo.release()
print("¡Proceso terminado! Archivos 'video_limpio.mp4' y 'comparativo_tiktok.mp4' generados.")


Procesados 30 fotogramas...
Procesados 60 fotogramas...
Procesados 90 fotogramas...
Procesados 120 fotogramas...
Procesados 150 fotogramas...
¡Proceso terminado! Archivos 'video_limpio.mp4' y 'comparativo_tiktok.mp4' generados.


In [ ]:
from IPython.display import HTML
from base64 import b64encode

# Recomprimir con ffmpeg para compatibilidad total con navegadores y móviles
!ffmpeg -y -i comparativo_tiktok.mp4 -vcodec libx264 -pix_fmt yuv420p comparativo_web.mp4 -loglevel quiet

mp4 = open("comparativo_web.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=400 controls autoplay loop>
      <source src="{data_url}" type="video/mp4">
</video>
""")
